##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

#### 這個notebook 是由[Nitin Tiwari](https://linkedin.com/in/tiwari-nitin) 創建的。

#### **社群連結：**
* [領英](https://linkedin.com/in/tiwari-nitin)
* [GitHub](https://github.com/NSTiwari)
* [推特](https://x.com/NSTiwari21)

# 影片中的零樣本目標檢測

此notebook指導您使用[PaliGemma](https://ai.google.dev/gemma/docs/paligemma)對影片執行零樣本目標檢測，並使用 OpenCV 和 PIL 繪製inference。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/PaliGemma/[PaliGemma_1]Zero_shot_object_detection_in_videos.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

### 取得 PaliGemma 的存取權限

在首次使用 PaliGemma 之前，你必須先透過 Hugging Face 申請模型存取權限，步驟如下：

1. 登入 [Hugging Face](https://huggingface.co)，如果你還沒有帳號，請先建立一個新的 Hugging Face 帳號。
2. 前往 [PaliGemma model card](https://huggingface.co/google/paligemma-3b-mix-224) 申請模型存取權限。
3. 填寫 consent form 並接受 terms and conditions。

若要建立 Hugging Face token，請開啟 Hugging Face 的 [**Settings** 頁面](https://huggingface.co/settings)，在左側選擇 **Access Tokens**，然後點選 **New token**。在接下來的視窗中，為 token 命名，並將類型設為 **Write** 以取得寫入權限。

接著，在 Colab 左側面板中選擇 **Secrets**（🔑），加入你的 Hugging Face token，並將其儲存為 `HF_TOKEN`。


### 選擇runtime

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 PaliGemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，按一下 **▾（其他連線選項）** 下拉式選單。
1. 選擇**更改 runtime 類型**。
1. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 第 1 步：安裝庫

In [1]:
!pip install bitsandbytes transformers accelerate peft -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 66.9 MB/s eta 0:00:00


### 步驟2：為Hugging Facetoken設定環境變量

In [4]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

### 步驟3：載入預訓練的PaliGemma基礎模型

In [5]:
from transformers import AutoTokenizer, PaliGemmaForConditionalGeneration, PaliGemmaProcessor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "google/paligemma-3b-mix-224"
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16)
processor = PaliGemmaProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/62.6k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/40.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

### 第四步：在影片上繪製inference的函數

In [6]:
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np

def draw_bounding_box(image, coordinates, label, width, height):
    global label_colors
    y1, x1, y2, x2 = coordinates
    y1, x1, y2, x2 = map(round, (y1*height, x1*width, y2*height, x2*width))

    text_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1, 3)
    text_width, text_height = text_size

    text_x = x1 + 2
    text_y = y1 - 5

    font_scale = 1
    label_rect_width = text_width + 8
    label_rect_height = int(text_height * font_scale)

    color = label_colors.get(label, None)
    if color is None:
        color = np.random.randint(0, 256, (3,)).tolist()
        label_colors[label] = color

    cv2.rectangle(image, (x1, y1 - label_rect_height), (x1 + label_rect_width, y1), color, -1)

    thickness = 2
    cv2.putText(image, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    return image

### 步驟5：設定輸入影片和文字prompt

In [7]:
input_video = 'input_video.mp4' # @param {type:"string"}

prompt = "detect person, phone, bottle" # @param {type: "string"}

output_file = 'output_video.avi' # @param {type: "string"}

### 步驟6：將輸入影片和文字prompt傳遞給PaliGemma並繪製inferences

In [ ]:
# Open the input video file.
cap = cv2.VideoCapture(input_video)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_file, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

label_colors = {}

while(True):
    ret, frame = cap.read()
    if not ret:
        break

    # Convert the frame to a PIL Image.
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # Send text prompt and image as input.
    inputs = processor(text=prompt, images=img,
                      padding="longest", do_convert_rgb=True, return_tensors="pt").to("cuda")
    model.to(device)
    inputs = inputs.to(dtype=model.dtype)

    # Get output.
    with torch.no_grad():
        output = model.generate(**inputs, max_length=496)

    paligemma_response = processor.decode(output[0], skip_special_tokens=True)[len(prompt):].lstrip("\n")
    detections = paligemma_response.split(" ; ")

    # Parse the output bounding box coordinates
    parsed_coordinates = []
    labels = []

    for item in detections:
        detection = item.replace("<loc", "").split()

        if len(detection) >= 2:
          coordinates_str = detection[0].replace(",", "")
          label = detection[1]
          if "<seg" in label:
            continue
          else:
            labels.append(label)
        else:
          # No label detected, skip the iteration.
          continue

        coordinates = coordinates_str.split(">")
        coordinates = coordinates[:4]

        if coordinates[-1] == '':
            coordinates = coordinates[:-1]


        coordinates = [int(coord)/1024 for coord in coordinates]
        parsed_coordinates.append(coordinates)

    width = img.size[0]
    height = img.size[1]

    # Draw bounding boxes on the frame
    image = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    for coordinates, label in zip(parsed_coordinates, labels):
      output_frame = draw_bounding_box(frame, coordinates, label, width, height)

    # Write the frame to the output video
    out.write(output_frame)

    # Exit on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture, output video writer, and destroy the window
cap.release()
out.release()
cv2.destroyAllWindows()
print("Output video " + output_file + " saved to disk.")

Output video output_video.avi saved to disk.
